In [162]:
import os 
import requests 
from langchain_core.tools import tool , InjectedToolArg
from typing import  Annotated


## Tool Creation 

Fetch Conversion Factor 

In [163]:
@tool
def get_convertion_factor(base_currency:str, target_currency:str)->float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    
    url = f"https://v6.exchangerate-api.com/v6/351110ff1e8c22c4fba28bd0/pair/{base_currency}/{target_currency}"

    response = requests.get(url)
    
    return response.json()

In [164]:
get_convertion_factor.invoke({'base_currency': 'USD','target_currency': 'PKR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1774569601,
 'time_last_update_utc': 'Fri, 27 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1774656001,
 'time_next_update_utc': 'Sat, 28 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'PKR',
 'conversion_rate': 279.675}

convert into currency 

In [165]:
@tool 
def convert(base_currency_value:int , convertionfactor : Annotated[float,InjectedToolArg] )->float:
    """
    given a currency conversion rate this function calculates the target currency value 
    from a given base currency value 
    """
    return f"Amount is {base_currency_value * convertionfactor}"

In [166]:
convert.invoke({"base_currency_value": 10 , 'convertionfactor' : 279.2897 })

'Amount is 2792.897'

# Tool Binding

- Setup Endpoint 

In [167]:
from langchain_core.messages import HumanMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
load_dotenv()

True

In [168]:
# 2. Check Token
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not token:
    print("Error: HUGGINGFACEHUB_API_TOKEN is not set in the .env file.")
else :
    print("Done")

Done


In [169]:
# 3. Setup the Endpoint
llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct", # A great open-source model for tool calling
    task="text-generation",
    temperature=0.1,
    max_new_tokens=512,
)

# 4. Wrap it in ChatHuggingFace so it can handle HumanMessages and Tools
llm = ChatHuggingFace(llm=llm_endpoint)

In [170]:
llm_with_tools = llm.bind_tools([get_convertion_factor,convert])

## Tool Calling

In [171]:
# 6. Test the Agent!
messages = [HumanMessage('What is the conversion factor between usd and pkr, and based on that can you convert 10 use to pkr')]
ai_message = llm_with_tools.invoke(messages)
print(ai_message)
print("-"*100)
# Print the output to see if the AI successfully decided to call your tool
print("AI Tool Decision:", ai_message.tool_calls)
messages.append(ai_message)

content='' additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"PKR"}', 'name': 'get_convertion_factor', 'description': None}, 'id': 'call_48kcgbuk4v7374r1g2bsv0gn', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'call_ko0k6q8fft1lw5oult633u7o', 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 348, 'total_tokens': 401}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d2dea-192b-7762-b072-de3433028c56-0' tool_calls=[{'name': 'get_convertion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': 'call_48kcgbuk4v7374r1g2bsv0gn', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_ko0k6q8fft1lw5oult633u7o', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'inpu

# Excution of Tool 

In [172]:
for i in ai_message.tool_calls:
    print(i)

{'name': 'get_convertion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': 'call_48kcgbuk4v7374r1g2bsv0gn', 'type': 'tool_call'}
{'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_ko0k6q8fft1lw5oult633u7o', 'type': 'tool_call'}


In [173]:

import json
for tool_call in ai_message.tool_calls:
    #excute 1st tool 
    if tool_call['name'] == "get_convertion_factor":
        tool_message_1 = get_convertion_factor.invoke(tool_call)
        #fetch coversion rate 
        conversion_rate = json.loads(tool_message_1.content)['conversion_rate']
        #append with message 
        messages.append(tool_message_1)

    #excute 2nd tool 
    if tool_call['name'] == 'convert':
        #fetch the current argument . 
        tool_call['args']['convertionfactor'] = conversion_rate
        tool_message_2 = convert.invoke(tool_call)
        messages.append(tool_message_2)

        

In [174]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and PKR is approximately 279.675. Based on this conversion factor, 10 USD is equivalent to 2796.75 PKR.'

In [179]:
messages

[HumanMessage(content='What is the conversion factor between usd and pkr, and based on that can you convert 10 use to pkr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"PKR"}', 'name': 'get_convertion_factor', 'description': None}, 'id': 'call_48kcgbuk4v7374r1g2bsv0gn', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'call_ko0k6q8fft1lw5oult633u7o', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 348, 'total_tokens': 401}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2dea-192b-7762-b072-de3433028c56-0', tool_calls=[{'name': 'get_convertion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': 'call_48kcgbuk4v7374r1g2bsv0gn',

In [181]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and PKR is approximately 279.675. Based on this conversion rate, 10 USD is equivalent to 2796.75 PKR.'